# Auto-Quant Experiment Analysis

Analysis of autonomous strategy-iteration results from `results.tsv`.

Run this after the agent has collected some experiments.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("results.tsv", sep="\t")
df["sharpe"] = pd.to_numeric(df["sharpe"], errors="coerce")
df["max_dd"] = pd.to_numeric(df["max_dd"], errors="coerce")
df["status"] = df["status"].str.strip().str.lower()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("keep", 0)
n_discard = counts.get("discard", 0)
n_crash = counts.get("crash", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")
    print(f"Crash rate: {n_crash}/{len(df)} = {n_crash / len(df):.1%}")

In [ ]:
# All KEPT experiments — the ones that stuck
kept = df[df["status"] == "keep"].copy().reset_index(drop=True)
print(f"KEPT experiments ({len(kept)}):\n")
for i, row in kept.iterrows():
    print(f"  #{i:3d}  sharpe={row['sharpe']:.4f}  dd={row['max_dd']:.1f}%  {row['description']}")

## Sharpe Frontier Over Time

Running max of sharpe as experiments accumulate. The plateau shows where
the agent stopped finding improvements.

In [ ]:
df_ordered = df.reset_index(drop=True)

# Running best must only consider KEEP experiments. Discarded runs
# (including retroactive discards — e.g. agent rolling back Goodhart
# wins) were explicitly rejected and should not move the frontier.
kept_sharpe = df_ordered["sharpe"].where(df_ordered["status"] == "keep")
df_ordered["running_max_sharpe"] = kept_sharpe.cummax()

status_color = {"keep": "tab:green", "discard": "tab:gray", "crash": "tab:red"}

fig, ax = plt.subplots(figsize=(10, 5))
for status, color in status_color.items():
    mask = df_ordered["status"] == status
    ax.scatter(df_ordered.index[mask], df_ordered["sharpe"][mask],
               alpha=0.5, c=color, label=status)
ax.plot(df_ordered.index, df_ordered["running_max_sharpe"],
        color="red", label="running best (keep-only)")
ax.set_xlabel("experiment #")
ax.set_ylabel("sharpe")
ax.set_title("Sharpe frontier")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Sharpe vs Drawdown Scatter

Any kept experiment with drawdown way worse than baseline is probably
over-fitting to a specific regime.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = {"keep": "green", "discard": "gray", "crash": "red"}
for status, g in df.groupby("status"):
    ax.scatter(g["max_dd"], g["sharpe"], alpha=0.6, label=status, c=colors.get(status, "black"))
ax.set_xlabel("max drawdown %")
ax.set_ylabel("sharpe")
ax.set_title("Sharpe vs drawdown, colored by outcome")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Description Word Frequency

What themes show up in the agent's own descriptions of what it tried?
A rough proxy for which directions it explored.

In [ ]:
from collections import Counter
import re

text = " ".join(df["description"].dropna().astype(str).str.lower().tolist())
words = re.findall(r"[a-z]{3,}", text)
stop = {"the", "and", "for", "with", "this", "that", "from", "was", "too", "add", "added", "use", "using"}
top = Counter(w for w in words if w not in stop).most_common(25)
print("Top words across all descriptions:")
for w, c in top:
    print(f"  {c:4d}  {w}")